# p=211, WD=0 — dense 1-step logging

Запуск Omnigrok для долгого контрольного эксперимента без weight decay. Все EDM- и projection-метрики записываются на каждом optimizer step. CSV пишется потоково, поэтому файл не пересобирается целиком на каждом шаге.

**Важно:** `log_every=1` существенно медленнее обычного режима: на каждом шаге выполняется мониторинговая оценка. Сначала рекомендуется проверить запуск на 1–2k шагов.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")
%cd /content/drive/MyDrive/grokking_prediction_original/2026-Project-202/code/Grokking/modular_addition_grokking_colab/wd0_dense_p211


In [ ]:
!pip install -q tensorboard


In [ ]:
# Reload the module if this cell was already run in the current runtime.
import importlib
import prime_sweep_omnigrok_dense as dense
dense = importlib.reload(dense)
Config, run_sweep = dense.Config, dense.run_sweep

CONFIG = Config(
    output_root="/content/drive/MyDrive/grokking_prime_sweep",
    protocol_name="omnigrok_p211_wd0_dense_500k_v1",
    primes=(211,),
    seeds=(42,),
    train_fraction=0.30,
    normalize_train_examples_per_class=True,
    train_examples_per_class=34.0,
    max_sampled_pairs=500_000,
    d_model=128, d_mlp=512, num_heads=4, d_head=32,
    batch_size=512, batch_size_by_p={211: 512}, model_dtype="float32",
    learning_rate=1e-3, betas=(0.9, 0.98),
    delayed_weight_decay=False, weight_decay=0.0, weight_decay_by_p={211: 0.0},
    # Final target step; a checkpoint at 500k resumes and runs to 1M.
    max_steps=1_000_000,
    log_every=1,
    diagnostic_every=1_000,
    checkpoint_every=10_000,
    tensorboard_enabled=True, tensorboard_flush_secs=5,
    tensorboard_histogram_every=5_000,
    text_log_enabled=True, text_log_filename="training.log", text_log_every=1_000,
    # Уменьшенный фиксированный monitor ускоряет dense-режим.
    monitor_train_pairs=512, monitor_val_pairs=512, eval_batch_size=512,
    projection_count=3,
    target_train_acc=0.99, target_val_acc=0.95,
    patience_logs=5, required_gap_steps=10_000, post_grok_steps=5_000,
    stream_csv=True, csv_flush_rows=100,
    # Keep the existing run and restore model, optimizer and counters.
    force_restart=False, resume_from_checkpoint=True,
    skip_completed=False, device="auto", fused_adamw=True,
)
CONFIG


## Live TensorBoard

Запустите до обучения. Графики обновляются примерно раз в 5 секунд.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/drive/MyDrive/grokking_prime_sweep/omnigrok_p211_wd0_dense_500k_v1 --reload_interval 5


## Обучение


In [ ]:
summaries = run_sweep(CONFIG)
summaries


In [ ]:
from pathlib import Path
import pandas as pd
run_root = Path(CONFIG.output_root) / CONFIG.protocol_name / "p_211" / "seed_42"
print("run directory:", run_root)
for name in ("training_log.csv", "training.log", "COMPLETED.json", "config.json"):
    path = run_root / name
    print(name, path.exists(), path)
if (run_root / "training_log.csv").exists():
    display(pd.read_csv(run_root / "training_log.csv").tail())
